# 3 — RESTORE per-image normalization

Fits KMeans/GMM/**SSC** per image × mutually-exclusive marker pair on the
REDSEA-corrected cells, sets each threshold at `mean+3σ` of the target-negative
population, guards degenerate per-image thresholds against the cohort median, and
writes `{m}_pos / {m}_norm / {m}_log2r` to `data/restore_gated_redsea/`.

Requires the vendored RESTORE submodule (`external/RESTORE`) and `spams` for SSC
(`pip install spams-bin`). The per-donor apply step runs across `n_jobs` processes.

In [ ]:
# Resolve the pipeline configuration (paths + params from ../config.ini).
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))  # make `phenocycler` importable from notebooks/
from phenocycler import load_config
cfg = load_config(pathlib.Path.cwd().parent / 'config.ini')
print('data_dir     :', cfg.data_dir)
print('images_dir   :', cfg.images_dir)
print('cells_csv    :', cfg.cells_csv)
print('n_jobs       :', cfg.n_jobs, '| use_gpu:', cfg.use_gpu)
cfg.discover_donors()  # donors found under data/cells/donor_id=* (empty until Step 1)

In [ ]:
from phenocycler.restore import run_restore

# Dry run first (1 image, thresholds+QC only):
# run_restore(cfg, limit_scenes=1, skip_apply=True)

run_restore(cfg, n_jobs=cfg.n_jobs)   # reads data/cells_redsea by default (SSC)

In [ ]:
import pandas as pd
thr = pd.read_csv(cfg.restore_thresholds_csv)
thr[thr.chosen].pivot(index='marker', columns='image', values='threshold').round(1)